# 🏗️ Notebook 1: Flash Sale — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/flash-sale
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

At 12:00:00 sharp, 1,000,000 users hit the site to buy one of 1,000 limited-edition items.
Stop the site from melting. Don't oversell.

### Functional requirements
- Sale starts at a fixed time.
- Enforce a hard stock cap (don't oversell).
- Each user buys at most **N** (e.g., 1–2).
- Charge payments only when allocated stock.

### Non-functional
- 100k–1M RPS for a few seconds.
- **No overselling** under any concurrency.
- Fair (FIFO-ish) — not literal FIFO, but "most users who tried in time get a chance."


## Architecture

```
   [1M clients]
        │ waiting-room page (static)
        ▼
   ┌────────────┐
   │ CDN/Edge   │ ← static "not open yet" page until 12:00
   └─────┬──────┘
         │ release
         ▼
   ┌────────────────┐
   │ Rate Limiter   │ ← token bucket per IP/user
   └─────┬──────────┘
         │
         ▼
   ┌────────────────┐
   │ Admission Q    │ ← bounded queue, drop on full
   │ (Kafka/Redis)  │
   └─────┬──────────┘
         │ consumers
         ▼
   ┌──────────────────────┐
   │ Stock Service        │ Redis DECR stock:itemX
   │ (atomic reservation) │
   └─────┬────────────────┘
         │ if reserved → eventually persist to DB
         ▼
   ┌──────────────┐       ┌────────────┐
   │ Payment Svc  │──────▶│ Orders DB  │
   └──────────────┘       └────────────┘
```

### Key design choices
1. **Front-load the filter**: CDN waiting room eliminates the thundering herd before it reaches origin.
2. **Rate limit** early (per user/IP).
3. **Bounded queue for admission**. Once queue is full → serve an "out of stock" page immediately.
4. **Atomic decrement in Redis** as the single source of truth for stock during the sale.
5. **Payment is async** — reserve first (cheap), charge later.
